# 06 · Hashing, Signatures, IDs & Text Utilities: `hashlib`, `hmac`, password hashing, `uuid`, `base64`, `textwrap`, `difflib`, `string`

**Module 12 – Standard Library Essentials** · Estimated time: 45 min

### 🎯 What you will learn
- Cryptographic hashes with `hashlib`: SHA-256, BLAKE2, file checksums
- Message authentication with **`hmac`** (signing webhooks, tokens)
- ✅ **Password hashing** done right (`hashlib.scrypt`, PBKDF2, and libraries like argon2/bcrypt), and why MD5/SHA-256 alone are wrong for passwords
- Unique identifiers with `uuid` (🆕 `uuid7`, time-ordered, Python 3.14)
- `base64` & `binascii` (recap)
- Text tools: `textwrap`, `difflib` (diffs & fuzzy matching), `string` constants & templates, `unicodedata`, `pprint`/`reprlib`

## 1. Hashes with `hashlib`

A **cryptographic hash** maps any data to a fixed-size "fingerprint". The same input always gives the same hash; a tiny change gives a completely different one; you can't reverse it.

In [1]:
import hashlib

print(hashlib.sha256(b"hello").hexdigest())
print(hashlib.sha256(b"hellp").hexdigest())          # one letter changed → totally different
print(hashlib.blake2b(b"hello", digest_size=16).hexdigest())
print(sorted(hashlib.algorithms_guaranteed))

2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
fdd7585e08c4e2afd71dcabdb4636c89d557a3f42db9e2040c8bbd1708aa4ce7
46fb7408d4f285228f4af516ea25851b
['blake2b', 'blake2s', 'md5', 'sha1', 'sha224', 'sha256', 'sha384', 'sha3_224', 'sha3_256', 'sha3_384', 'sha3_512', 'sha512', 'shake_128', 'shake_256']


### File checksums (verify downloads, detect duplicates/changes)

In [2]:
from pathlib import Path

def file_sha256(path, chunk_size=65_536):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):      # stream big files
            h.update(chunk)
    return h.hexdigest()

Path("a.txt").write_text("same content")
Path("b.txt").write_text("same content")
Path("c.txt").write_text("different")
print({p: file_sha256(p)[:16] for p in ("a.txt", "b.txt", "c.txt")})
with open("a.txt", "rb") as f:
    print(hashlib.file_digest(f, "sha256").hexdigest()[:16])   # 3.11+ helper does the chunking for you

{'a.txt': 'a636bd7cd42060a4', 'b.txt': 'a636bd7cd42060a4', 'c.txt': '9d6f965ac832e40a'}
a636bd7cd42060a4


| Algorithm | Use? |
|---|---|
| MD5, SHA-1 | ❌ broken for security (collisions); OK only for non-security checksums/caching keys |
| SHA-256 / SHA-512 / SHA-3 | ✅ integrity, signatures, content addressing |
| BLAKE2 | ✅ fast and secure; supports keys and salts |

## 2. HMAC: proving a message wasn't tampered with

A hash proves *integrity*; an **HMAC** (hash + secret key) also proves the message came from someone who knows the key. Used for webhook signatures (GitHub, Stripe), signed cookies, API request signing.

In [3]:
import hmac, secrets, json

SECRET = secrets.token_bytes(32)           # shared between sender and receiver

def sign(payload: dict) -> str:
    body = json.dumps(payload, sort_keys=True).encode()
    return hmac.new(SECRET, body, hashlib.sha256).hexdigest()

def verify(payload: dict, signature: str) -> bool:
    return hmac.compare_digest(sign(payload), signature)     # constant-time comparison!

order = {"id": 1042, "amount": 499}
sig = sign(order)
print(sig[:20], "...", verify(order, sig))
order["amount"] = 1                         # attacker changes the amount
print("tampered order valid?", verify(order, sig))

f4ade10741d300257b54 ... True
tampered order valid? False


> ⚠️ Compare secrets with `hmac.compare_digest` (or `secrets.compare_digest`), never `==`: `==` stops at the first different character, leaking timing information.

## 3. ✅ Password hashing

**Never store passwords in plain text, and never with a plain fast hash** (`sha256(password)`): attackers can try billions of guesses per second, and identical passwords give identical hashes. Use a **slow, salted, memory-hard** key-derivation function:

In [4]:
import os

def hash_password(password: str) -> str:
    salt = os.urandom(16)
    key = hashlib.scrypt(password.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
    return f"scrypt${salt.hex()}${key.hex()}"

def check_password(password: str, stored: str) -> bool:
    _, salt_hex, key_hex = stored.split("$")
    key = hashlib.scrypt(password.encode(), salt=bytes.fromhex(salt_hex), n=2**14, r=8, p=1, dklen=32)
    return hmac.compare_digest(key.hex(), key_hex)

stored = hash_password("correct horse battery staple")
print(stored[:60], "...")
print(check_password("correct horse battery staple", stored), check_password("guess", stored))
print(hash_password("same") != hash_password("same"))          # random salt → different hashes

scrypt$68a4f578afb7e28a945cffab06f10a8b$7c404415c18804802e28 ...
True False
True


In production prefer a dedicated library: **`argon2-cffi`** (Argon2id, the current recommendation), **`bcrypt`**, or your framework's helpers (Django's `make_password`). `hashlib.pbkdf2_hmac` is the standard-library fallback.

## 4. Unique IDs with `uuid`

In [5]:
import uuid

print(uuid.uuid4())                         # random: the usual choice
u7 = [uuid.uuid7() for _ in range(3)]       # 🆕 3.14: time-ordered (sortable) → great database keys
print(*u7, sep="\n")
print(u7 == sorted(u7))
print(uuid.uuid5(uuid.NAMESPACE_URL, "https://python.org"))   # deterministic from a name
u = uuid.uuid4()
print(u.hex, u.version, uuid.UUID(str(u)) == u)

c9ab483f-3940-4684-9bd8-88ed59cfe4f7
01a159a9-d91a-71e1-b857-46d622bea321
01a159a9-d91a-71e1-b857-46d71cd852e2
01a159a9-d91a-71e1-b857-46d8f838388c
True
54456a86-ff7a-507c-ab55-c4d535450fb7
6014d06ba7824c6ba5912a77cc46671e 4 True


| Version | Based on | Use |
|---|---|---|
| `uuid1` | time + MAC address | ⚠️ leaks the machine's MAC |
| `uuid3` / `uuid5` | namespace + name hash | stable IDs from names |
| `uuid4` | random | general purpose |
| `uuid7` 🆕 | Unix-time + random | **sortable** IDs, better database index locality |
| `uuid8` 🆕 | custom | vendor-specific layouts |

For short random tokens use `secrets.token_urlsafe()` (Module 12 · 03).

## 5. `base64` (recap) and data URLs

In [6]:
import base64
png_bytes = Path("blue_color.png").read_bytes()
data_url = "data:image/png;base64," + base64.b64encode(png_bytes).decode()
print(data_url[:60], "...", len(data_url), "chars")      # embed small images in HTML/CSS

data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAMgAAADIAQMAAA ... 162 chars


## 6. Text utilities

### `textwrap`

In [7]:
import textwrap

para = ("Python is a high-level, general-purpose programming language. Its design philosophy "
        "emphasizes code readability with the use of significant indentation.")
print(textwrap.fill(para, width=50))
print(textwrap.shorten(para, width=60, placeholder=" [...]"))
print(textwrap.indent("line 1\nline 2", "> "))
print(textwrap.dedent("""
        indented
          more
    """).strip())

Python is a high-level, general-purpose
programming language. Its design philosophy
emphasizes code readability with the use of
significant indentation.
Python is a high-level, general-purpose programming [...]
> line 1
> line 2
indented
  more


### `difflib`: diffs and fuzzy matching

In [8]:
import difflib

old = ["name = 'app'", "version = '1.0'", "debug = False"]
new = ["name = 'app'", "version = '1.1'", "debug = False", "workers = 4"]
print("\n".join(difflib.unified_diff(old, new, "config_old", "config_new", lineterm="")))

commands = ["install", "uninstall", "list", "freeze", "show"]
print(difflib.get_close_matches("instal", commands))          # "did you mean ...?"
print(round(difflib.SequenceMatcher(None, "colour", "color").ratio(), 2))

--- config_old
+++ config_new
@@ -1,3 +1,4 @@
 name = 'app'
-version = '1.0'
+version = '1.1'
 debug = False
+workers = 4
['install', 'uninstall', 'list']
0.91


### `string` constants, `Template`, and `unicodedata`

In [9]:
import string, unicodedata

print(string.ascii_letters, string.digits, string.punctuation, sep="\n")
print(string.Template("Hello $name, you owe $$${amount}").substitute(name="Asha", amount=50))
print(string.capwords("the quick brown fox"))

for ch in "é½😀":
    print(ch, unicodedata.name(ch), unicodedata.category(ch))

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
print(strip_accents("Crème brûlée à São Paulo"))
print("ﬁ" == "fi", unicodedata.normalize("NFKC", "ﬁ") == "fi")      # normalise before comparing

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
0123456789
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Hello Asha, you owe $50
The Quick Brown Fox
é LATIN SMALL LETTER E WITH ACUTE Ll
½ VULGAR FRACTION ONE HALF No
😀 GRINNING FACE So
Creme brulee a Sao Paulo
False True


### `pprint` and `reprlib`

In [10]:
import pprint, reprlib
data = {"users": [{"name": f"user{i}", "tags": ["a", "b"]} for i in range(3)], "count": 3}
pprint.pp(data, width=60, depth=2)
print(reprlib.repr(list(range(1000))))          # abbreviated repr for huge objects

{'users': [{...}, {...}, {...}], 'count': 3}
[0, 1, 2, 3, 4, 5, ...]


## 📝 Summary
- `hashlib` for integrity (SHA-256/BLAKE2), never MD5/SHA-1 for security.
- `hmac` + `compare_digest` to sign and verify messages.
- Passwords: salted slow KDFs (`scrypt`, Argon2, bcrypt), never plain hashes.
- `uuid4` for random IDs; 🆕 `uuid7` for sortable database keys.
- `textwrap`, `difflib`, `string`, `unicodedata`, `pprint`, `reprlib` handle everyday text tasks.

## 🏋️ Practice
1. Find duplicate files in a folder by hashing their contents.
2. Verify a "GitHub webhook": compute `sha256=` + HMAC of a body with a secret and compare with a header value.
3. Build a "did you mean?" helper for a CLI using `difflib.get_close_matches`.
4. Generate 5 `uuid7`s with small sleeps between them and confirm they sort in creation order.